In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import optuna
import wandb


In [2]:
#Settings
# =======================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


STYLE_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\data\Post_Impressionism"
CONTENT_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\pics"

# Loading AlexNet tarined model to act as a judge
JUDGE_MODEL_PATH = "best_vangogh_alexnet_final.pth"

NUM_IMAGES_TO_TEST = 3
SEARCH_IMG_SIZE = 224
NUM_STEPS = 300

print(f"✅ Device: {DEVICE}")

✅ Device: cuda


In [3]:


#  AlexNet Extractor

print("🎨 Loading AlexNet Feature Extractor...")
# Loading the alexNet and taking it features to act as the "painter"
alexnet_extractor = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).features.to(DEVICE).eval()
for param in alexnet_extractor.parameters():
    param.requires_grad_(False)


def load_judge_model(path):
    print(f"⚖️ Loading AlexNet Judge from {path}...")
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    # התאמת הראש בדיוק כמו באימון
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, 2)

    try:
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        print("✅ Judge loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading judge: {e}")

        exit()

    model.to(DEVICE).eval()
    for param in model.parameters():
        param.requires_grad = False
    return model


if os.path.exists(JUDGE_MODEL_PATH):
    judge_model = load_judge_model(JUDGE_MODEL_PATH)
else:
    print(f"⚠️ Warning: Judge file {JUDGE_MODEL_PATH} not found.")
    exit()

🎨 Loading AlexNet Feature Extractor...
⚖️ Loading AlexNet Judge from best_vangogh_alexnet_final.pth...
✅ Judge loaded successfully.


In [4]:
# ==========================================
def load_image_tensor(img_path, max_size=SEARCH_IMG_SIZE):
    try:
        image = Image.open(img_path).convert('RGB')
        in_transform = transforms.Compose([
            transforms.Resize((max_size, max_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
        return in_transform(image)[:3, :, :].unsqueeze(0).to(DEVICE)
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return None

In [5]:
def get_image_paths(directory, is_style=False, limit=3):
    if not os.path.exists(directory): return []
    all_files = os.listdir(directory)
    valid_images = []
    for f in all_files:
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        if is_style:
            if "vincent-van-gogh" in f.lower():
                valid_images.append(os.path.join(directory, f))
        else:
            valid_images.append(os.path.join(directory, f))

    random.shuffle(valid_images)
    return valid_images[:limit]

In [6]:
def get_features(image, model, layers_dict):
    features = {}
    x = image
    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers_dict:
            features[layers_dict[name]] = x
    return features

In [7]:
def gram_matrix(tensor):
    b, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    return torch.mm(tensor, tensor.t())

In [8]:
# Style transfer function
# ==========================================
def run_style_transfer_alexnet(content_tensor, style_tensor, c_weight, s_weight, tv_weight, lr,
                               layer_weights_dict, content_layer_name, num_steps=NUM_STEPS):
    if content_tensor.shape != style_tensor.shape:
        style_tensor = F.interpolate(style_tensor, size=content_tensor.shape[-2:], mode='bilinear')

    target = content_tensor.clone().requires_grad_(True).to(DEVICE)
    optimizer = optim.Adam([target], lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

    # *** מיפוי השכבות של AlexNet ***
    # 0=Conv1, 3=Conv2, 6=Conv3, 8=Conv4, 10=Conv5
    layers_map = {
        '0': 'conv1',
        '3': 'conv2',
        '6': 'conv3',
        '8': 'conv4',
        '10': 'conv5'
    }

    # חישוב פיצ'רים ראשוני
    content_features = get_features(content_tensor, alexnet_extractor, layers_map)
    style_features = get_features(style_tensor, alexnet_extractor, layers_map)
    style_grams = {k: gram_matrix(v) for k, v in style_features.items() if k in layer_weights_dict}

    for i in range(num_steps):
        target_features = get_features(target, alexnet_extractor, layers_map)

        # 1. Content Loss
        # וודא שהשכבה קיימת במיפוי
        if content_layer_name in target_features:
            c_loss = torch.mean((target_features[content_layer_name] - content_features[content_layer_name]) ** 2)
        else:
            c_loss = 0  # Fallback

        # 2. Style Loss
        s_loss = 0
        for layer_name, weight in layer_weights_dict.items():
            if layer_name in target_features:
                target_gram = gram_matrix(target_features[layer_name])
                style_gram = style_grams[layer_name]
                b, d, h, w = target_features[layer_name].shape

                layer_loss = weight * torch.mean((target_gram - style_gram) ** 2)
                s_loss += layer_loss / (d * h * w)

        # 3. TV Loss
        diff_i = torch.sum(torch.abs(target[:, :, :, 1:] - target[:, :, :, :-1]))
        diff_j = torch.sum(torch.abs(target[:, :, 1:, :] - target[:, :, :-1, :]))
        tv_loss = (diff_i + diff_j) / (target.nelement())

        total_loss = c_weight * c_loss + s_weight * s_loss + tv_weight * tv_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()

    return target

In [9]:
# Data preparation
# ==========================================
print("\n--- Preparing Data for AlexNet Search ---")
content_paths = get_image_paths(CONTENT_DIR, is_style=False, limit=NUM_IMAGES_TO_TEST)
style_paths = get_image_paths(STYLE_DIR, is_style=True, limit=NUM_IMAGES_TO_TEST)

content_tensors = [load_image_tensor(p) for p in content_paths]
style_tensors = [load_image_tensor(p) for p in style_paths]
content_tensors = [t for t in content_tensors if t is not None]
style_tensors = [t for t in style_tensors if t is not None]

min_len = min(len(content_tensors), len(style_tensors))
content_tensors = content_tensors[:min_len]
style_tensors = style_tensors[:min_len]

print(f"✅ Ready on {min_len} pairs.")


--- Preparing Data for AlexNet Search ---
✅ Ready on 3 pairs.


In [10]:
# 6. Objective Function (AlexNet Version)
# ==========================================
def objective(trial):
    # A. Global Params
    c_weight = 1.0
    s_weight = trial.suggest_float("style_weight", 1e4, 1e9, log=True)
    lr = trial.suggest_float("lr", 0.01, 0.1)
    tv_weight = trial.suggest_float("tv_weight", 1e-6, 1e-3, log=True)

    # B. Content Layer Selection
    content_layer_name = trial.suggest_categorical("content_layer", ['conv3', 'conv4', 'conv5'])

    # C. Style Layer Weights
    w1 = trial.suggest_float("w_conv1", 0.1, 1.0)
    w2 = trial.suggest_float("w_conv2", 0.1, 1.0)
    w3 = trial.suggest_float("w_conv3", 0.1, 1.0)
    w4 = trial.suggest_float("w_conv4", 0.1, 1.0)
    w5 = trial.suggest_float("w_conv5", 0.1, 1.0)

    current_layer_weights = {
        'conv1': w1, 'conv2': w2, 'conv3': w3, 'conv4': w4, 'conv5': w5
    }

    total_score = 0.0

    for i in range(len(content_tensors)):
        ct = content_tensors[i]
        st = style_tensors[i]

        # performing style transfer
        gen_img = run_style_transfer_alexnet(
            ct, st, c_weight, s_weight, tv_weight, lr,
            layer_weights_dict=current_layer_weights,
            content_layer_name=content_layer_name,
            num_steps=NUM_STEPS
        )

        with torch.no_grad():
            output = judge_model(gen_img)
            # Prob for being van gogh painting
            prob = F.softmax(output, dim=1)[0, 1].item()

        total_score += prob

    avg_score = total_score / len(content_tensors)

    # Reporting to WanDB
    wandb.log({
        "trial": trial.number,
        "score": avg_score,
        "style_weight": s_weight,
        "lr": lr,
        "content_layer": content_layer_name,
        "w_conv1": w1, "w_conv2": w2, "w_conv3": w3
    })

    print(f"Trial {trial.number}: Score={avg_score:.4f} | StyleW={s_weight:.2e} | Content={content_layer_name}")

    return avg_score

In [11]:
# main
# ==========================================
if __name__ == "__main__":
    # wandb.login(key="...")

    print("🚀 Starting AlexNet Style transfer Hyperparameter Search...")

    # שם פרויקט חדש ב-W&B
    wandb.init(project="vangogh-style-transfer-search", name="optuna_search_alexnet", reinit=True)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)  # תריץ 30 ניסויים כמו ב-VGG

    print("\n" + "=" * 50)
    print("🏆 ALEXNET SEARCH FINISHED")
    print(f"Best Judge Score: {study.best_value:.4f}")
    print("Best Params:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    print("=" * 50)

    wandb.finish()

🚀 Starting AlexNet Hyperparameter Search...


wandb: Currently logged in as: guygalanti (guygalanti-tel-aviv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[I 2026-01-18 13:57:49,443] A new study created in memory with name: no-name-4defaa03-7138-4f92-89ce-c7fb908debf4


Trial 0: Score=0.9542 | StyleW=5.22e+04 | Content=conv5


[I 2026-01-18 13:57:55,601] Trial 0 finished with value: 0.9542181293169657 and parameters: {'style_weight': 52220.17238319586, 'lr': 0.092165088286406, 'tv_weight': 0.0003481258648001121, 'content_layer': 'conv5', 'w_conv1': 0.8021025736328294, 'w_conv2': 0.8158795613778688, 'w_conv3': 0.1681888649871181, 'w_conv4': 0.8289781727205277, 'w_conv5': 0.12248291851898997}. Best is trial 0 with value: 0.9542181293169657.


...

[I 2026-01-18 14:00:25,299] Trial 28 finished with value: 0.9672388633092245 and parameters: {'style_weight': 352675.7811828086, 'lr': 0.07614265063384257, 'tv_weight': 1.8478463015843655e-05, 'content_layer': 'conv4', 'w_conv1': 0.5776760898806044, 'w_conv2': 0.26213159467378977, 'w_conv3': 0.9250342376286386, 'w_conv4': 0.5176413481484069, 'w_conv5': 0.37465307523788904}. Best is trial 21 with value: 0.9756548404693604.


Trial 29: Score=0.9574 | StyleW=7.42e+04 | Content=conv4


[I 2026-01-18 14:00:30,846] Trial 29 finished with value: 0.9573656320571899 and parameters: {'style_weight': 74203.00798683344, 'lr': 0.0640897153368724, 'tv_weight': 2.295486147458028e-06, 'content_layer': 'conv4', 'w_conv1': 0.8372735713944836, 'w_conv2': 0.7482466649267037, 'w_conv3': 0.9945037957868221, 'w_conv4': 0.8382270444430953, 'w_conv5': 0.15991562134284293}. Best is trial 21 with value: 0.9756548404693604.



🏆 ALEXNET SEARCH FINISHED
Best Judge Score: 0.9757
Best Params:
  style_weight: 70751.93516147001
  lr: 0.06413264328164367
  tv_weight: 2.838423296733179e-06
  content_layer: conv4
  w_conv1: 0.8735399075864216
  w_conv2: 0.2796787925793863
  w_conv3: 0.9398442338185024
  w_conv4: 0.724796164829389
  w_conv5: 0.290539223473126


lr,█▄▇▆█▇▇▅▂▇▁▃▃▃▂▄▂▄▅▅▅▅▅▆▅▆▅▄▆▅
score,▅▆▄▃▄▃▅▆▆▁▁▇▄▆▆▅▅▇▇█▇█▇▇▇▃▇▆▇▅
style_weight,▁▃▁▂▁▁▂█▁▂▁▇▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁
trial,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
w_conv1,▇▃█▄▂▂▄▇▃█▅▅▅▅▃▁▄▆▆▇▇▇▆▇█▆▇▆▅▇
w_conv2,▇▃▆▇▄█▇▂▅█▅▁▂▃▅▆▁▃▃▃▂▃▄▂▃▄▂▃▂▇
w_conv3,▁▄▇▇▄▇▃▅▂▂▁▅▅▆▂▃▅▆▆█████▆▆▇▆▇█
content_layer,conv4
lr,0.06409
score,0.95737
style_weight,74203.00799
